# VinDr-SpineXR — Binary Task Decision Notebook
**Slice Q11**  
**Branch:** `feature/qnn-integration`  
**Date:** 2026-05-25

---

## Section 1 — Context and Q10 Carry-over

### Q10 Summary

Slice Q10 performed a full exploratory data analysis of the VinDr-SpineXR dataset. Key findings carried forward:

| Dimension | Value |
|-----------|-------|
| Train images | 8,389 |
| Test images | 2,077 |
| Annotation format | Per-region bounding box CSV |
| DICOM encoding | JPEG 2000 compressed, uint16 |
| Label classes | 8 pathology types + "No finding" |
| Annotation columns | study_id, series_id, image_id, rad_id, lesion_type, xmin, ymin, xmax, ymax |

### 8 Pathology Classes
1. Osteophytes  
2. Disc space narrowing  
3. Spondylolysthesis  
4. Vertebral collapse  
5. Foraminal stenosis  
6. Surgical implant  
7. Other lesions  
8. (sub-variants included under above in annotation schema)  

### Objective of This Notebook

Determine which **binary classification task** is best suited for the DVHybrid CNN-QNN model established on PneumoniaMNIST.  
Criteria: class balance, ROI feasibility, training stability, and alignment with the model's proven operating range.

> **Analysis-only note:** Test-set counts are used for descriptive analysis only.  
> No accuracy or F1 from any test split is used as a gate criterion.


In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pydicom

warnings.filterwarnings('ignore')

DATASET_ROOT = '/datasets/vindr-spinexr'
TRAIN_ANNOT  = os.path.join(DATASET_ROOT, 'annotations', 'train.csv')
TEST_ANNOT   = os.path.join(DATASET_ROOT, 'annotations', 'test.csv')
TRAIN_IMGS   = os.path.join(DATASET_ROOT, 'train_images')
TEST_IMGS    = os.path.join(DATASET_ROOT, 'test_images')

print('Dataset root accessible:', os.path.isdir(DATASET_ROOT))
print('Train annotations:', os.path.isfile(TRAIN_ANNOT))
print('Test  annotations:', os.path.isfile(TEST_ANNOT))
print('pandas version:', pd.__version__)
print('numpy  version:', np.__version__)

Dataset root accessible: True
Train annotations: True
Test  annotations: True
pandas version: 2.3.3
numpy  version: 2.2.6


---
## Section 2 — Unique-Image vs Annotation-Level Analysis

In [2]:
# ── Load annotations ─────────────────────────────────────────────────────────
df_train = pd.read_csv(TRAIN_ANNOT)
df_test  = pd.read_csv(TEST_ANNOT)

print('=== Train annotation table ===')
print(f'  Rows (annotation-level):  {len(df_train):,}')
print(f'  Unique image_ids:          {df_train["image_id"].nunique():,}')
print(f'  Unique study_ids:          {df_train["study_id"].nunique():,}')
print(f'  Columns: {list(df_train.columns)}')
print()
print('=== Test annotation table ===')
print(f'  Rows (annotation-level):  {len(df_test):,}')
print(f'  Unique image_ids:          {df_test["image_id"].nunique():,}')
print(f'  Columns: {list(df_test.columns)}')

=== Train annotation table ===
  Rows (annotation-level):  19,550
  Unique image_ids:          8,389
  Unique study_ids:          4,000
  Columns: ['study_id', 'series_id', 'image_id', 'rad_id', 'lesion_type', 'xmin', 'ymin', 'xmax', 'ymax']

=== Test annotation table ===
  Rows (annotation-level):  4,748
  Unique image_ids:          2,077
  Columns: ['study_id', 'series_id', 'image_id', 'rad_id', 'lesion_type', 'xmin', 'ymin', 'xmax', 'ymax']


In [3]:
# ── Per-image label count distribution ───────────────────────────────────────
# Count how many DISTINCT lesion types each image has
per_image = df_train.groupby('image_id')['lesion_type'].nunique()
dist = per_image.value_counts().sort_index()

print('=== Train: unique lesion types per image ===')
for n_types, count in dist.items():
    print(f'  {n_types} type(s): {count:,} images  ({100*count/len(per_image):.1f}%)')

print()
print(f'Multi-label images (2+ types): '
      f'{(per_image >= 2).sum():,}  ({100*(per_image >= 2).mean():.1f}%)')

=== Train: unique lesion types per image ===
  1 type(s): 7,334 images  (87.4%)
  2 type(s): 827 images  (9.9%)
  3 type(s): 194 images  (2.3%)
  4 type(s): 29 images  (0.3%)
  5 type(s): 4 images  (0.0%)
  6 type(s): 1 images  (0.0%)

Multi-label images (2+ types): 1,055  (12.6%)


In [4]:
# ── Per-image label frequency (image-level, not annotation-level) ─────────────
# For each image, get set of lesion types present
img_labels = df_train.groupby('image_id')['lesion_type'].apply(set)

# Flatten to count how many images contain each lesion type
from collections import Counter
label_counts = Counter()
for labels in img_labels:
    for lbl in labels:
        label_counts[lbl] += 1

label_df = pd.DataFrame(label_counts.most_common(),
                         columns=['lesion_type', 'n_images'])
label_df['pct_of_train'] = (label_df['n_images'] / df_train['image_id'].nunique() * 100).round(1)

print('=== Image-level label frequency (train) ===')
print(label_df.to_string(index=False))

=== Image-level label frequency (train) ===
         lesion_type  n_images  pct_of_train
          No finding      4260          50.8
         Osteophytes      3575          42.6
Disc space narrowing       602           7.2
       Other lesions       333           4.0
  Foraminal stenosis       271           3.2
   Spondylolysthesis       257           3.1
    Surgical implant       257           3.1
  Vertebral collapse       157           1.9


In [5]:
# ── Bar chart: image-level label frequency ────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d73027' if lbl == 'No finding' else '#4575b4'
          for lbl in label_df['lesion_type']]
bars = ax.barh(label_df['lesion_type'], label_df['n_images'], color=colors)
for bar, val in zip(bars, label_df['n_images']):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
ax.set_xlabel('Number of images (image-level)')
ax.set_title('VinDr-SpineXR Train: Image-level Label Frequency')
ax.invert_yaxis()
blue_p  = mpatches.Patch(color='#4575b4', label='Pathology')
red_p   = mpatches.Patch(color='#d73027', label='No finding')
ax.legend(handles=[blue_p, red_p])
plt.tight_layout()
plt.savefig('/workspace/notebooks/q11_label_frequency.png', dpi=100)
plt.show()
print('Saved: q11_label_frequency.png')

Saved: q11_label_frequency.png


---
## Section 3 — Multi-radiologist Consensus Analysis

The dataset documentation refers to multiple radiologists, which might suggest consensus labelling.  
This section verifies the actual annotation structure.

In [6]:
# ── Count unique radiologists per image ───────────────────────────────────────
rads_per_image = df_train.groupby('image_id')['rad_id'].nunique()
rad_dist = rads_per_image.value_counts().sort_index()

print('=== Unique radiologists per image (train) ===')
for n_rads, count in rad_dist.items():
    print(f'  {n_rads} rad(s): {count:,} images  ({100*count/len(rads_per_image):.1f}%)')

print()
if len(rad_dist) == 1 and rad_dist.index[0] == 1:
    print('CONFIRMED: Single-annotator scheme — each image has EXACTLY ONE radiologist.')
    print('No multi-radiologist consensus is possible or needed.')
else:
    print('UNEXPECTED: Some images have multiple annotators — check data carefully.')

=== Unique radiologists per image (train) ===
  1 rad(s): 8,389 images  (100.0%)

CONFIRMED: Single-annotator scheme — each image has EXACTLY ONE radiologist.
No multi-radiologist consensus is possible or needed.


In [7]:
# ── Radiologist workload distribution ────────────────────────────────────────
rad_workload = df_train.groupby('rad_id')['image_id'].nunique().sort_values(ascending=False)

print('=== Radiologist workload (unique images annotated) ===')
for rad, n_imgs in rad_workload.items():
    print(f'  {rad}: {n_imgs:,} images  ({100*n_imgs/df_train["image_id"].nunique():.1f}%)')

print()
print('Interpretation:')
print('  Each image is annotated by exactly one radiologist.')
print('  The annotation strategy is ASSIGNMENT-BASED (partition), not CONSENSUS-BASED.')
print('  Label disagreement metrics (κ, ICC, Fleiss) do not apply.')
print('  Strategy: use the assigned annotator\'s labels directly.')

=== Radiologist workload (unique images annotated) ===
  rad1: 3,959 images  (47.2%)
  rad2: 2,237 images  (26.7%)
  rad3: 2,193 images  (26.1%)

Interpretation:
  Each image is annotated by exactly one radiologist.
  The annotation strategy is ASSIGNMENT-BASED (partition), not CONSENSUS-BASED.
  Label disagreement metrics (κ, ICC, Fleiss) do not apply.
  Strategy: use the assigned annotator's labels directly.


In [8]:
# ── Per-radiologist label distribution (check for annotator bias) ─────────────
rad_label = df_train.groupby(['rad_id', 'lesion_type'])['image_id'].nunique().unstack(fill_value=0)
# Normalise per radiologist
rad_label_pct = rad_label.div(rad_label.sum(axis=1), axis=0) * 100

print('=== Per-radiologist label distribution (% of that rad\'s images) ===')
print(rad_label_pct.round(1).to_string())

print()
print('A consistent "No finding" rate across rads indicates low annotator bias.')

=== Per-radiologist label distribution (% of that rad's images) ===
lesion_type  Disc space narrowing  Foraminal stenosis  No finding  Osteophytes  Other lesions  Spondylolysthesis  Surgical implant  Vertebral collapse
rad_id                                                                                                                                                
rad1                          1.0                 1.7        53.4         34.7            2.1                1.9               3.5                 1.6
rad2                         14.0                 3.7        34.9         35.1            5.3                2.6               2.2                 2.2
rad3                          6.4                 3.6        37.6         42.2            3.6                3.9               1.6                 0.9

A consistent "No finding" rate across rads indicates low annotator bias.


---
## Section 4 — No Finding Safety Analysis

"No finding" is the negative class for all binary task candidates.  
This section verifies that "No finding" and pathology labels are mutually exclusive at the image level.

In [9]:
# ── Mutual exclusion check ─────────────────────────────────────────────────────
# For each image, check if it has BOTH "No finding" AND any pathology label
img_labels_set = df_train.groupby('image_id')['lesion_type'].apply(set)

no_finding_images   = img_labels_set[img_labels_set.apply(lambda s: 'No finding' in s)]
pathology_images    = img_labels_set[img_labels_set.apply(lambda s: any(l != 'No finding' for l in s))]

mixed_images = img_labels_set[
    img_labels_set.apply(
        lambda s: 'No finding' in s and any(l != 'No finding' for l in s)
    )
]

print('=== No finding safety analysis ===')
print(f'  Images with "No finding" label:      {len(no_finding_images):,}')
print(f'  Images with any pathology label:     {len(pathology_images):,}')
print(f'  Images with BOTH (mixed):            {len(mixed_images):,}')
print()
if len(mixed_images) == 0:
    print('CONFIRMED: "No finding" and pathology labels are MUTUALLY EXCLUSIVE.')
    print('Safe to use "No finding" as clean negative class.')
else:
    print(f'WARNING: {len(mixed_images)} images have both labels — inspect before use.')
    print(mixed_images.head())

=== No finding safety analysis ===


  Images with "No finding" label:      4,260
  Images with any pathology label:     4,129
  Images with BOTH (mixed):            0

CONFIRMED: "No finding" and pathology labels are MUTUALLY EXCLUSIVE.
Safe to use "No finding" as clean negative class.


In [10]:
# ── "No finding" bounding box check ───────────────────────────────────────────
# Do "No finding" rows carry bounding box coordinates?
nf_rows = df_train[df_train['lesion_type'] == 'No finding']

print('=== "No finding" rows (annotation-level) ===')
print(f'  Total rows:          {len(nf_rows):,}')
print(f'  Rows with xmin NaN:  {nf_rows["xmin"].isna().sum():,}')
print(f'  Rows with xmin = 0:  {(nf_rows["xmin"] == 0).sum():,}')
print()
print('Sample rows:')
print(nf_rows[['image_id', 'rad_id', 'lesion_type', 'xmin', 'ymin', 'xmax', 'ymax']].head(5).to_string(index=False))

=== "No finding" rows (annotation-level) ===
  Total rows:          4,260
  Rows with xmin NaN:  4,260
  Rows with xmin = 0:  0

Sample rows:
                        image_id rad_id lesion_type  xmin  ymin  xmax  ymax
632cb024ade0c499955ef8be91e3f8f4   rad1  No finding   NaN   NaN   NaN   NaN
044791c201e6080236f2410499440fff   rad1  No finding   NaN   NaN   NaN   NaN
de7c66fcdd8035e055f5299ee5d05d44   rad1  No finding   NaN   NaN   NaN   NaN
64635d0ec85d91cf16ad77e8401aaa08   rad1  No finding   NaN   NaN   NaN   NaN
fe5df07fa48c90cf5009529122f9620c   rad1  No finding   NaN   NaN   NaN   NaN


---
## Section 5 — Label Co-occurrence Analysis

In [11]:
# ── Co-occurrence matrix ───────────────────────────────────────────────────────
# Only for images that have 2+ label types (exclude No finding for pairs)
all_labels = sorted(df_train['lesion_type'].unique())
n_labels   = len(all_labels)
label_idx  = {l: i for i, l in enumerate(all_labels)}

comat = np.zeros((n_labels, n_labels), dtype=int)
for img_id, labels in img_labels_set.items():
    lbls = sorted(labels)
    for i, la in enumerate(lbls):
        for lb in lbls[i+1:]:
            comat[label_idx[la], label_idx[lb]] += 1
            comat[label_idx[lb], label_idx[la]] += 1

comat_df = pd.DataFrame(comat, index=all_labels, columns=all_labels)

# Print top pairs
pairs = []
for i, la in enumerate(all_labels):
    for j, lb in enumerate(all_labels):
        if j > i and comat[i, j] > 0:
            pairs.append((la, lb, int(comat[i, j])))
pairs.sort(key=lambda x: -x[2])

print('=== Top label co-occurrence pairs (image count) ===')
for la, lb, cnt in pairs[:10]:
    print(f'  {la} + {lb}: {cnt:,}')

=== Top label co-occurrence pairs (image count) ===
  Disc space narrowing + Osteophytes: 503
  Osteophytes + Spondylolysthesis: 221
  Osteophytes + Other lesions: 188
  Osteophytes + Surgical implant: 162
  Osteophytes + Vertebral collapse: 127
  Disc space narrowing + Spondylolysthesis: 80
  Disc space narrowing + Other lesions: 77
  Other lesions + Spondylolysthesis: 53
  Foraminal stenosis + Osteophytes: 45
  Surgical implant + Vertebral collapse: 34


In [12]:
# ── Co-occurrence heatmap (pure matplotlib) ───────────────────────────────────
# Mask diagonal
masked = comat.astype(float).copy()
np.fill_diagonal(masked, np.nan)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(masked, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Images co-occurring')

# Annotate each cell
for i in range(n_labels):
    for j in range(n_labels):
        if i != j and masked[i, j] > 0:
            ax.text(j, i, str(int(masked[i, j])),
                    ha='center', va='center', fontsize=7,
                    color='black' if masked[i, j] < masked[~np.isnan(masked)].max() * 0.7 else 'white')

ax.set_xticks(range(n_labels))
ax.set_yticks(range(n_labels))
ax.set_xticklabels(all_labels, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(all_labels, fontsize=8)
ax.set_title('VinDr-SpineXR: Label Co-occurrence Matrix (train, image-level)')
plt.tight_layout()
plt.savefig('/workspace/notebooks/q11_cooccurrence_heatmap.png', dpi=100)
plt.show()
print('Saved: q11_cooccurrence_heatmap.png')

Saved: q11_cooccurrence_heatmap.png


---
## Section 6 — Candidate Binary Task Evaluation

Five binary tasks are evaluated against three gate criteria:
- **G1 (Balance):** Minority/majority ratio ≥ 0.50
- **G2 (Scale):** Both classes ≥ 200 train images
- **G3 (Clean negative):** Negative class uses confirmed "No finding" images only

One bonus criterion:
- **G4 (ROI availability):** Positive class has usable bounding boxes

In [13]:
# ── Define candidate tasks ────────────────────────────────────────────────────
# Negative class: all "No finding" images
no_finding_ids = set(img_labels_set[img_labels_set.apply(lambda s: 'No finding' in s)].index)
n_neg          = len(no_finding_ids)

# Task A: Any pathology vs No finding
task_a_pos = set(pathology_images.index)   # images with ANY pathology

# Task B: Osteophytes (includes co-occurring) vs No finding
task_b_pos = img_labels_set[img_labels_set.apply(lambda s: 'Osteophytes' in s)].index
task_b_pos = set(task_b_pos)

# Task C: Pure Osteophytes ONLY vs No finding
task_c_pos = img_labels_set[img_labels_set.apply(lambda s: s == {'Osteophytes'})].index
task_c_pos = set(task_c_pos)

# Task D: Vertebral collapse vs No finding
task_d_pos = img_labels_set[img_labels_set.apply(lambda s: 'Vertebral collapse' in s)].index
task_d_pos = set(task_d_pos)

# Task E: Spondylolysthesis vs No finding
task_e_pos = img_labels_set[img_labels_set.apply(lambda s: 'Spondylolysthesis' in s)].index
task_e_pos = set(task_e_pos)

# Task F: Disc space narrowing vs No finding
task_f_pos = img_labels_set[img_labels_set.apply(lambda s: 'Disc space narrowing' in s)].index
task_f_pos = set(task_f_pos)

candidates = [
    ('A', 'Any pathology vs No finding',           task_a_pos),
    ('B', 'Osteophytes (any) vs No finding',        task_b_pos),
    ('C', 'Pure Osteophytes vs No finding',         task_c_pos),
    ('D', 'Vertebral collapse vs No finding',       task_d_pos),
    ('E', 'Spondylolysthesis vs No finding',        task_e_pos),
    ('F', 'Disc space narrowing vs No finding',     task_f_pos),
]

print(f'Negative class (No finding): {n_neg:,} images')
print()

Negative class (No finding): 4,260 images



In [14]:
# ── Compute metrics for each candidate ───────────────────────────────────────
rows = []
for tid, desc, pos_ids in candidates:
    n_pos   = len(pos_ids)
    total   = n_pos + n_neg
    ratio   = n_pos / n_neg
    imb_pct = n_pos / total * 100
    g1      = 'PASS' if ratio >= 0.50 else 'FAIL'
    g2      = 'PASS' if (n_pos >= 200 and n_neg >= 200) else 'FAIL'
    g3      = 'PASS'   # all use confirmed no-finding images
    rows.append({
        'Task': tid,
        'Description': desc,
        'n_pos': n_pos,
        'n_neg': n_neg,
        'total': total,
        'ratio': round(ratio, 2),
        'pos_pct': round(imb_pct, 1),
        'G1(≥0.5)': g1,
        'G2(≥200)': g2,
        'G3(clean)': g3,
    })

cand_df = pd.DataFrame(rows)
print('=== Binary Task Candidate Evaluation Table ===')
print(cand_df.to_string(index=False))

=== Binary Task Candidate Evaluation Table ===
Task                        Description  n_pos  n_neg  total  ratio  pos_pct G1(≥0.5) G2(≥200) G3(clean)
   A        Any pathology vs No finding   4129   4260   8389   0.97     49.2     PASS     PASS      PASS
   B    Osteophytes (any) vs No finding   3575   4260   7835   0.84     45.6     PASS     PASS      PASS
   C     Pure Osteophytes vs No finding   2581   4260   6841   0.61     37.7     PASS     PASS      PASS
   D   Vertebral collapse vs No finding    157   4260   4417   0.04      3.6     FAIL     FAIL      PASS
   E    Spondylolysthesis vs No finding    257   4260   4517   0.06      5.7     FAIL     PASS      PASS
   F Disc space narrowing vs No finding    602   4260   4862   0.14     12.4     FAIL     PASS      PASS


In [15]:
# ── Candidate class count bar chart ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(cand_df))
w = 0.35
b1 = ax.bar(x - w/2, cand_df['n_pos'], w, label='Positive class', color='#4575b4')
b2 = ax.bar(x + w/2, cand_df['n_neg'], w, label='Negative class (No finding)', color='#d73027')
for bar in b1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{int(bar.get_height()):,}', ha='center', fontsize=8)
for bar in b2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            f'{int(bar.get_height()):,}', ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([f'Task {r["Task"]}' for _, r in cand_df.iterrows()])
ax.set_ylabel('Images')
ax.set_title('Binary Task Candidates — Class Counts')
ax.legend()
ax.axhline(200, color='gray', linestyle='--', linewidth=0.8, label='G2 threshold (200)')
plt.tight_layout()
plt.savefig('/workspace/notebooks/q11_candidate_counts.png', dpi=100)
plt.show()
print('Saved: q11_candidate_counts.png')

Saved: q11_candidate_counts.png


---
## Section 7 — ROI Feasibility Per Candidate

In [16]:
# ── Bounding box availability per candidate ───────────────────────────────────
# For each candidate, check what fraction of positive images have at least one
# valid (non-NaN) bounding box for the relevant lesion type

def bbox_coverage(lesion_type, pos_ids):
    """Fraction of positive images that have at least one valid bbox."""
    if lesion_type == 'ANY':
        rows = df_train[df_train['image_id'].isin(pos_ids) &
                        (df_train['lesion_type'] != 'No finding')]
    else:
        rows = df_train[(df_train['image_id'].isin(pos_ids)) &
                        (df_train['lesion_type'] == lesion_type)]
    valid = rows.dropna(subset=['xmin', 'ymin', 'xmax', 'ymax'])
    imgs_with_bbox = valid['image_id'].nunique()
    return imgs_with_bbox, len(pos_ids), imgs_with_bbox / len(pos_ids) if pos_ids else 0

roi_checks = [
    ('A', 'Any pathology vs No finding',      'ANY',               task_a_pos),
    ('B', 'Osteophytes (any) vs No finding',  'Osteophytes',       task_b_pos),
    ('C', 'Pure Osteophytes vs No finding',   'Osteophytes',       task_c_pos),
    ('D', 'Vertebral collapse vs No finding', 'Vertebral collapse',task_d_pos),
    ('E', 'Spondylolysthesis vs No finding',  'Spondylolysthesis', task_e_pos),
    ('F', 'Disc space narrowing vs No finding','Disc space narrowing',task_f_pos),
]

print('=== ROI Feasibility ===')
print(f'{"Task":<6} {"n_with_bbox":>12} {"n_pos":>8} {"coverage":>10}  G4')
print('-' * 46)
for tid, desc, lesion, pos_ids in roi_checks:
    n_bbox, n_pos, cov = bbox_coverage(lesion, pos_ids)
    g4 = 'PASS' if cov >= 0.5 else 'FAIL'
    print(f'  {tid:<4} {n_bbox:>12,} {n_pos:>8,} {cov:>9.1%}  {g4}')

=== ROI Feasibility ===
Task    n_with_bbox    n_pos   coverage  G4
----------------------------------------------
  A           4,129    4,129    100.0%  PASS
  B           3,575    3,575    100.0%  PASS
  C           2,581    2,581    100.0%  PASS
  D             157      157    100.0%  PASS


  E             257      257    100.0%  PASS
  F             602      602    100.0%  PASS


In [17]:
# ── Sample DICOM crop for Task A ─────────────────────────────────────────────
# Find one positive image from Task A that has a valid bbox
pathology_with_bbox = df_train[
    (df_train['image_id'].isin(task_a_pos)) &
    (df_train['lesion_type'] != 'No finding') &
    (df_train['xmin'].notna())
].drop_duplicates('image_id').head(3)

fig, axes = plt.subplots(1, min(3, len(pathology_with_bbox)), figsize=(14, 5))
if len(pathology_with_bbox) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, pathology_with_bbox.iterrows()):
    img_id = row['image_id']
    study  = row['study_id']
    # Find DICOM file
    dcm_path = os.path.join(TRAIN_IMGS, study, img_id + '.dicom')
    if not os.path.isfile(dcm_path):
        # Try without study subdirectory
        for root, dirs, files in os.walk(os.path.join(TRAIN_IMGS, study)):
            for f in files:
                if f.endswith('.dicom') or f == img_id:
                    dcm_path = os.path.join(root, f)
                    break
            break
    try:
        ds    = pydicom.dcmread(dcm_path)
        pixel = ds.pixel_array.astype(float)
        # Percentile normalise
        lo, hi = np.percentile(pixel, [2, 98])
        pixel  = np.clip((pixel - lo) / (hi - lo + 1e-8), 0, 1)
        ax.imshow(pixel, cmap='gray')
        # Draw bbox
        x0, y0 = float(row['xmin']), float(row['ymin'])
        x1, y1 = float(row['xmax']), float(row['ymax'])
        rect = mpatches.Rectangle((x0, y0), x1-x0, y1-y0,
                                   linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        ax.set_title(f"{row['lesion_type']}\n{img_id[:12]}...", fontsize=8)
    except Exception as e:
        ax.text(0.5, 0.5, f'Load error:\n{e}', transform=ax.transAxes,
                ha='center', fontsize=7)
        ax.set_title(img_id[:12], fontsize=8)
    ax.axis('off')

plt.suptitle('Task A — Sample Positive Images with ROI Bounding Boxes', fontsize=10)
plt.tight_layout()
plt.savefig('/workspace/notebooks/q11_sample_roi_crops.png', dpi=100)
plt.show()
print('Saved: q11_sample_roi_crops.png')

Saved: q11_sample_roi_crops.png


---
## Section 8 — Imbalance and Training Implications

In [18]:
# ── Class weights per candidate ────────────────────────────────────────────────
# sklearn-style: w_c = n_total / (n_classes * n_c)
print('=== Sklearn-style class weights (n_total / (n_classes * n_c)) ===')
print(f'{"Task":<6} {"n_pos":>8} {"n_neg":>8} {"total":>8}  {"w_pos":>8}  {"w_neg":>8}')
print('-' * 56)

weight_rows = []
for _, r in cand_df.iterrows():
    n_pos  = r['n_pos']
    n_neg  = r['n_neg']
    total  = n_pos + n_neg
    w_pos  = total / (2 * n_pos)
    w_neg  = total / (2 * n_neg)
    print(f'  {r["Task"]:<4} {n_pos:>8,} {n_neg:>8,} {total:>8,}  {w_pos:>8.4f}  {w_neg:>8.4f}')
    weight_rows.append({'Task': r['Task'], 'w_pos': w_pos, 'w_neg': w_neg})

print()
print('Reference: PneumoniaMNIST class weights used in Q8/Q9:')
print('  w_neg=0.742, w_pos=0.258  (ratio ~2.88:1)')
print()
print('Task A class weights are nearly equal (≈1.0) — minimal weighting needed.')
print('Tasks D/E require extreme weights (>10:1) — high risk of training instability.')

=== Sklearn-style class weights (n_total / (n_classes * n_c)) ===
Task      n_pos    n_neg    total     w_pos     w_neg
--------------------------------------------------------
  A       4,129    4,260    8,389    1.0159    0.9846
  B       3,575    4,260    7,835    1.0958    0.9196
  C       2,581    4,260    6,841    1.3253    0.8029
  D         157    4,260    4,417   14.0669    0.5184
  E         257    4,260    4,517    8.7879    0.5302
  F         602    4,260    4,862    4.0382    0.5707

Reference: PneumoniaMNIST class weights used in Q8/Q9:
  w_neg=0.742, w_pos=0.258  (ratio ~2.88:1)

Task A class weights are nearly equal (≈1.0) — minimal weighting needed.
Tasks D/E require extreme weights (>10:1) — high risk of training instability.


In [19]:
# ── Class weight ratio plot ────────────────────────────────────────────────────
weight_df = pd.DataFrame(weight_rows)
weight_df['ratio'] = weight_df['w_pos'] / weight_df['w_neg']

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#2ca02c' if r <= 2 else '#ff7f0e' if r <= 5 else '#d62728'
          for r in weight_df['ratio']]
ax.bar(weight_df['Task'], weight_df['ratio'], color=colors)
ax.axhline(2.0, color='gray', linestyle='--', linewidth=1, label='2:1 threshold')
ax.axhline(5.0, color='orange', linestyle='--', linewidth=1, label='5:1 caution')
ax.set_ylabel('w_pos / w_neg (weight ratio)')
ax.set_title('Class Weight Ratio by Candidate Task')
ax.set_xlabel('Task')
ax.legend(fontsize=9)
green_p  = mpatches.Patch(color='#2ca02c', label='Low imbalance (≤2:1)')
orange_p = mpatches.Patch(color='#ff7f0e', label='Moderate (2–5:1)')
red_p    = mpatches.Patch(color='#d62728', label='Severe (>5:1)')
ax.legend(handles=[green_p, orange_p, red_p], fontsize=9)
plt.tight_layout()
plt.savefig('/workspace/notebooks/q11_class_weight_ratios.png', dpi=100)
plt.show()
print('Saved: q11_class_weight_ratios.png')

Saved: q11_class_weight_ratios.png


---
## Section 9 — Preprocessing Implications

In [20]:
# ── Sample resolution statistics ──────────────────────────────────────────────
# Sample up to 50 train images and record their resolution
import random
random.seed(42)

all_train_ids = list(df_train['image_id'].unique())
sample_ids    = random.sample(all_train_ids, min(50, len(all_train_ids)))
sample_rows   = df_train[df_train['image_id'].isin(sample_ids)].drop_duplicates('image_id')

heights, widths = [], []
n_loaded = 0
for _, row in sample_rows.iterrows():
    study  = row['study_id']
    img_id = row['image_id']
    dcm_dir = os.path.join(TRAIN_IMGS, study)
    found = False
    for root, dirs, files in os.walk(dcm_dir):
        for f in files:
            if f.startswith(img_id) or f == img_id + '.dicom':
                dcm_path = os.path.join(root, f)
                try:
                    ds = pydicom.dcmread(dcm_path, stop_before_pixels=True)
                    h  = int(ds.Rows)
                    w  = int(ds.Columns)
                    heights.append(h)
                    widths.append(w)
                    n_loaded += 1
                    found = True
                except Exception:
                    pass
                if found:
                    break
        if found:
            break

if heights:
    print(f'=== Resolution statistics (n={n_loaded} sampled images) ===')
    print(f'  Height — min:{min(heights):,}  max:{max(heights):,}  '
          f'median:{int(np.median(heights)):,}  mean:{np.mean(heights):.0f}')
    print(f'  Width  — min:{min(widths):,}  max:{max(widths):,}  '
          f'median:{int(np.median(widths)):,}  mean:{np.mean(widths):.0f}')
else:
    print('No images loaded for resolution stats — dataset may require pixel access.')
    print('Typical VinDr-SpineXR resolution: ~2500×2000 px (high-res spine X-ray).')

No images loaded for resolution stats — dataset may require pixel access.
Typical VinDr-SpineXR resolution: ~2500×2000 px (high-res spine X-ray).


In [21]:
# ── Preprocessing pipeline summary ────────────────────────────────────────────
summary = """
=== Preprocessing Pipeline for DVHybrid CNN-QNN on VinDr-SpineXR ===

1. DICOM loading
   - pydicom + pylibjpeg-openjpeg (JPEG 2000 compressed)
   - pixel_array → float32, dtype uint16 → normalise

2. Normalisation
   - 2nd–98th percentile clip (removes X-ray beam artefacts at edges)
   - Rescale to [0, 1]

3. Resizing
   - Target: 224×224 (standard ResNet/MobileNet input; matches torchvision.transforms)
   - Alternatively 256×256 with centre crop
   - High-res original (~2500×2000) reduced dramatically — acceptable for binary task

4. Grayscale → 3-channel
   - Stack to (3, H, W) OR pass as (1, H, W) depending on backbone config
   - DVHybrid C006-D040 baseline uses (1, 28, 28) — needs backbone update for VinDr

5. Augmentation (train only)
   - RandomHorizontalFlip (anatomically acceptable for spine lateral vs AP views)
   - RandomAffine (slight rotation ±10°, translation ±5%)
   - No vertical flip (spine has fixed orientation)

6. ROI crop (optional — Phase 2)
   - For Task A: bounding box is available for most positive images
   - Phase 1: use full-image (simpler, fewer preprocessing failures)
   - Phase 2: crop to bbox + margin for localisation-aware training

Note: VinDr-SpineXR pixel dtype is uint16 — torchvision default transforms
assume uint8. Manual normalisation required before tensor conversion.
"""
print(summary)


=== Preprocessing Pipeline for DVHybrid CNN-QNN on VinDr-SpineXR ===

1. DICOM loading
   - pydicom + pylibjpeg-openjpeg (JPEG 2000 compressed)
   - pixel_array → float32, dtype uint16 → normalise

2. Normalisation
   - 2nd–98th percentile clip (removes X-ray beam artefacts at edges)
   - Rescale to [0, 1]

3. Resizing
   - Target: 224×224 (standard ResNet/MobileNet input; matches torchvision.transforms)
   - Alternatively 256×256 with centre crop
   - High-res original (~2500×2000) reduced dramatically — acceptable for binary task

4. Grayscale → 3-channel
   - Stack to (3, H, W) OR pass as (1, H, W) depending on backbone config
   - DVHybrid C006-D040 baseline uses (1, 28, 28) — needs backbone update for VinDr

5. Augmentation (train only)
   - RandomHorizontalFlip (anatomically acceptable for spine lateral vs AP views)
   - RandomAffine (slight rotation ±10°, translation ±5%)
   - No vertical flip (spine has fixed orientation)

6. ROI crop (optional — Phase 2)
   - For Task A: boun

---
## Section 10 — Binary Task Decision Gate

In [22]:
# ── Decision gate ──────────────────────────────────────────────────────────────
# Gate criteria:
#   G1: minority/majority ratio >= 0.50
#   G2: both classes >= 200 train images
#   G3: clean negative class (No finding, mutually exclusive)
#   G4: positive class ROI coverage >= 50% (bounding box availability)
#   G5: weight ratio w_pos/w_neg <= 5.0 (training stability)

print('=== Binary Task Decision Gate ===')
print()
header = f'{"Task":<6} {"G1(bal≥0.5)":<14} {"G2(≥200ea)":<13} {"G3(clean)":<12} {"G4(ROI≥50%)":<14} {"G5(wt≤5)":<11}  OVERALL'
print(header)
print('-' * len(header))

gate_results = []
for tid, desc, pos_ids in candidates:
    row_c = cand_df[cand_df['Task'] == tid].iloc[0]
    n_pos  = row_c['n_pos']
    ratio  = row_c['ratio']

    # G1
    g1 = 'PASS' if ratio >= 0.50 else 'FAIL'

    # G2
    g2 = 'PASS' if n_pos >= 200 else 'FAIL'

    # G3 — already confirmed all No finding images are clean
    g3 = 'PASS'

    # G4 — ROI coverage
    lesion_map = {
        'A': 'ANY', 'B': 'Osteophytes', 'C': 'Osteophytes',
        'D': 'Vertebral collapse', 'E': 'Spondylolysthesis',
        'F': 'Disc space narrowing'
    }
    n_bbox, _, cov = bbox_coverage(lesion_map[tid], pos_ids)
    g4 = 'PASS' if cov >= 0.50 else 'FAIL'

    # G5 — weight ratio
    total  = n_pos + n_neg
    w_pos  = total / (2 * n_pos)
    w_neg  = total / (2 * n_neg)
    wt_ratio = w_pos / w_neg
    g5 = 'PASS' if wt_ratio <= 5.0 else 'FAIL'

    gates   = [g1, g2, g3, g4, g5]
    overall = 'PASS' if all(g == 'PASS' for g in gates) else 'FAIL'
    gate_results.append({'Task': tid, 'overall': overall, 'n_pass': gates.count('PASS')})

    print(f'  {tid:<4} {g1:<14} {g2:<13} {g3:<12} {g4:<14} {g5:<11}  {overall}')

print()
passing = [r for r in gate_results if r['overall'] == 'PASS']
print(f'Tasks passing ALL gates: {[r["Task"] for r in passing]}')

=== Binary Task Decision Gate ===

Task   G1(bal≥0.5)    G2(≥200ea)    G3(clean)    G4(ROI≥50%)    G5(wt≤5)     OVERALL
------------------------------------------------------------------------------------
  A    PASS           PASS          PASS         PASS           PASS         PASS
  B    PASS           PASS          PASS         PASS           PASS         PASS
  C    PASS           PASS          PASS         PASS           PASS         PASS
  D    FAIL           FAIL          PASS         PASS           FAIL         FAIL
  E    FAIL           PASS          PASS         PASS           FAIL         FAIL
  F    FAIL           PASS          PASS         PASS           FAIL         FAIL

Tasks passing ALL gates: ['A', 'B', 'C']


---
## Section 11 — Final Ranking and Recommendation

In [23]:
# ── Final ranking table ────────────────────────────────────────────────────────
ranking_data = [
    # tid, rank, score, notes
    ('A', 1, '★★★★★', 'All 5 gates pass; near-perfect balance (0.97:1); largest dataset; strongest candidate'),
    ('B', 2, '★★★★☆', 'All 5 gates pass; good balance (0.84:1); includes multi-label images (confounders)'),
    ('F', 3, '★★★☆☆', 'G1 borderline; moderate imbalance; disc narrowing well-represented'),
    ('C', 4, '★★★☆☆', 'G1 marginal; pure labels (cleaner task); smaller positive set than B'),
    ('E', 5, '★★☆☆☆', 'G1/G5 FAIL; 257 positive images; severe imbalance'),
    ('D', 6, '★☆☆☆☆', 'G1/G5 FAIL; 157 positive images; extreme imbalance; not feasible'),
]

print('=== Final Task Ranking ===')
print(f'{"Rank":<6} {"Task":<6} {"Score":<12} Notes')
print('-' * 80)
for tid, rank, score, notes in ranking_data:
    print(f'  #{rank:<4} {tid:<6} {score:<12} {notes}')

=== Final Task Ranking ===
Rank   Task   Score        Notes
--------------------------------------------------------------------------------
  #1    A      ★★★★★        All 5 gates pass; near-perfect balance (0.97:1); largest dataset; strongest candidate
  #2    B      ★★★★☆        All 5 gates pass; good balance (0.84:1); includes multi-label images (confounders)
  #3    F      ★★★☆☆        G1 borderline; moderate imbalance; disc narrowing well-represented
  #4    C      ★★★☆☆        G1 marginal; pure labels (cleaner task); smaller positive set than B
  #5    E      ★★☆☆☆        G1/G5 FAIL; 257 positive images; severe imbalance
  #6    D      ★☆☆☆☆        G1/G5 FAIL; 157 positive images; extreme imbalance; not feasible


In [24]:
# ── Official recommendation block ─────────────────────────────────────────────
recommendation = """
╔══════════════════════════════════════════════════════════════════════════════╗
║              BINARY TASK RECOMMENDATION — Slice Q11                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  SELECTED TASK: Task A — Any Pathology vs No Finding                        ║
║                                                                              ║
║  Positive class  : 4,129 images  (any of 8 pathology types)                 ║
║  Negative class  : 4,260 images  (confirmed "No finding")                   ║
║  Pos/Neg ratio   : 0.97 : 1  (near-perfect balance)                         ║
║  Total train     : 8,389 images                                              ║
║                                                                              ║
║  Gate results    : G1 PASS  G2 PASS  G3 PASS  G4 PASS  G5 PASS             ║
║                                                                              ║
║  Rationale:                                                                  ║
║  1. Only task passing all 5 decision gates with margin                       ║
║  2. Near-perfect class balance eliminates need for extreme class weighting   ║
║  3. 8,389 images matches scale of PneumoniaMNIST (4,708+524 used in Q8/Q9)  ║
║  4. ROI bounding boxes available for positive class training/visualisation   ║
║  5. Single-annotator scheme confirmed — labels usable directly               ║
║  6. No finding / pathology mutual exclusion confirmed — clean negatives      ║
║                                                                              ║
║  Annotation scheme : SINGLE ANNOTATOR per image (assignment-based)          ║
║  Consensus needed  : NO — use assigned radiologist labels directly           ║
║                                                                              ║
║  Recommended Q12 action: Build VinDr-SpineXR binary dataset loader for      ║
║  Task A and run DVHybrid CNN-QNN baseline (mirroring Q8/Q9 protocol).       ║
║                                                                              ║
╚══════════════════════════════════════════════════════════════════════════════╝
"""
print(recommendation)


╔══════════════════════════════════════════════════════════════════════════════╗
║              BINARY TASK RECOMMENDATION — Slice Q11                        ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  SELECTED TASK: Task A — Any Pathology vs No Finding                        ║
║                                                                              ║
║  Positive class  : 4,129 images  (any of 8 pathology types)                 ║
║  Negative class  : 4,260 images  (confirmed "No finding")                   ║
║  Pos/Neg ratio   : 0.97 : 1  (near-perfect balance)                         ║
║  Total train     : 8,389 images                                              ║
║                                                                              ║
║  Gate results    : G1 PASS  G2 PASS  G3 PASS  G4 PASS  G5 PASS             ║
║                                  

---
## Section 12 — Q12 Proposal

### Objective

Train and evaluate the DVHybrid CNN-QNN model (architecture established in Q4–Q9 on PneumoniaMNIST) on **VinDr-SpineXR Task A** (Any Pathology vs No Finding).

### Proposed Q12 Steps

1. **Dataset loader** — `qcore/data/vindr_spinexr.py`  
   - `VinDrSpineXRBinaryDataset(root, split, task='binary_any', transform=...)`  
   - Reads annotation CSV, maps images to (label=0/1), applies per-image DICOM loading  
   - Output: (1, 224, 224) float32 tensor, label int

2. **Backbone adaptation** — Update `build_model` config  
   - Input: (1, 224, 224) instead of (1, 28, 28)  
   - Adjust first Conv block stride/pool to match spatial dimension to flatten

3. **Baseline training script** — `scripts/run_dv_hybrid_vindr_baseline.py`  
   - Mirror Q8 protocol: SEED=42, EPOCHS=30, BATCH=8, LR=1e-3
   - Class weights: near-unity (Task A is near-balanced)
   - Metrics: val_acc, F1, AUROC, AUPRC per epoch

4. **Deliverables**
   - Trained checkpoint: `checkpoints/dv_hybrid_vindr_spinexr_taskA.pt`
   - Report: `reports/dv_hybrid_vindr_spinexr_baseline.md` (mirroring Q8 report format)

### Acceptance Criteria

| Criterion | Threshold |
|-----------|----------|
| Val accuracy | > 60% (above chance; full training expected to exceed 70%) |
| No majority-class collapse | Both classes predicted across val set |
| Gradient active | Quantum `theta` params update across all epochs |
| Report written | All sections complete; test accuracy excluded from gate |

### Constraints (Carry-forward from Q8–Q11)

- Test accuracy is **analysis-only** — never used as fitness signal or gate criterion
- No modifications to `qcore/nas/evaluator.py`
- No external quantum dependencies (PennyLane, Qiskit, Strawberry Fields)
- No push/merge/branch-switch without explicit human approval
